<a href="https://colab.research.google.com/github/stella1298/AIFFEL_Quest_EPA/blob/master/LLM/LLM05/RAG_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# [실습 환경 준비]
# 라이브러리 버전을 지정해 설치합니다. 버전을 지정하지 않으면 최신 버전이 설치되어
# 함수 이름이나 인자가 달라져 이 코드가 동작하지 않을 수 있습니다.
!pip install -q "ragas==0.4.*" "langchain==0.3.*" "langchain-core==0.3.*" "langchain-community==0.3.*" "langchain-openai==0.3.*" datasets nest_asyncio

# RAGAS는 평가 속도를 높이기 위해 내부적으로 asyncio.gather를 사용해 수백 개의 API를 동시에 비동기 호출합니다.
# 노트북 환경은 이미 자체적인 비동기 이벤트 루프를 돌리고 있어서 충돌이 발생합니다.
# nest_asyncio는 이 충돌을 막아주어 노트북에서도 RAGAS가 병렬 처리를 할 수 있게 해줍니다.
#2.비동기 처리 설정 # Colab 환경에서 RAGAS의 비동기 호출 충돌을 방지
import nest_asyncio
nest_asyncio.apply()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.2/353.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0

In [6]:
import os
from getpass import getpass
from google.colab import userdata

# 입력창에 키를 붙여 넣으면 이 세션의 환경 변수로만 저장됩니다. (파일에는 남지 않아요.)
#os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키를 붙여 넣으세요: ")
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

print("OpenAI API Key 설정 완료")

OpenAI API Key 설정 완료


In [7]:
#3.RAGAS 평가 모델 설정
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# RAGAS 평가를 담당할 LLM
evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Answer Relevancy 등의 평가에 사용할 임베딩 모델
evaluator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("평가 모델 설정 완료")

평가 모델 설정 완료


In [8]:
#4. RAGAS Metric 불러오기
from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

print("RAGAS 평가 지표 준비 완료")

RAGAS 평가 지표 준비 완료


/tmp/ipykernel_645/3477666927.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_645/3477666927.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_645/3477666927.py:4: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_645/3477666927.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.

In [9]:
#5. 먼저 가장 간단한 RAG 데이터 만들기
#테스트 데이터
questions = ["주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?"]

ground_truths = [
    "주가수익비율(PER)은 기업의 주가를 주당순이익(EPS)으로 나눈 값으로, "
    "기업의 주가가 이익에 비해 어느 정도 수준인지 나타내는 지표입니다."
]

answers = [
    "PER(Price Earnings Ratio)은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. "
    "EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다. "
    "PER이 낮으면 일반적으로 이익에 비해 주가가 낮은 것으로 해석할 수 있지만, "
    "업종 특성과 기업의 성장성 등을 함께 고려해야 합니다."
]

contexts = [[
    "주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. "
    "기업의 수익성 대비 주가 수준을 평가하는 지표입니다.",

    "EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.",

    "업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 "
    "개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다."
]]

In [10]:
#6. RAGAS Dataset 만들기

from datasets import Dataset

data = {
    "user_input": questions,
    "retrieved_contexts": contexts,
    "response": answers,
    "reference": ground_truths
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['user_input', 'retrieved_contexts', 'response', 'reference'],
    num_rows: 1
})


In [11]:
display(dataset.to_pandas())

,user_input,retrieved_contexts,response,reference
0,주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?,[주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. ...,PER(Price Earnings Ratio)은 현재 주가를 주당순이익(EPS)으로...,"주가수익비율(PER)은 기업의 주가를 주당순이익(EPS)으로 나눈 값으로, 기업의 ..."


In [12]:
#7. 첫 번째 RAGAS 평가
result = evaluate(
    dataset=dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False
)

print(result)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

{'context_precision': 1.0000, 'context_recall': 1.0000, 'faithfulness': 0.5000, 'answer_relevancy': 0.0000}


In [16]:
#8. 평가 결과를 DataFrame으로 확인
from datasets import Dataset
import pandas as pd

df = result.to_pandas()

pd.set_option("display.max_colwidth", None)

display(df)

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?,"[주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. 기업의 수익성 대비 주가 수준을 평가하는 지표입니다., EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다., 업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다.]","PER(Price Earnings Ratio)은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다. PER이 낮으면 일반적으로 이익에 비해 주가가 낮은 것으로 해석할 수 있지만, 업종 특성과 기업의 성장성 등을 함께 고려해야 합니다.","주가수익비율(PER)은 기업의 주가를 주당순이익(EPS)으로 나눈 값으로, 기업의 주가가 이익에 비해 어느 정도 수준인지 나타내는 지표입니다.",1.0,1.0,0.5,0.0


In [17]:
metrics = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

display(df[metrics])

,faithfulness,answer_relevancy,context_precision,context_recall
0,0.5,0.0,1.0,1.0


In [18]:
#9. "테스트" 추가 : RAG 테스트 함수
def test_rag(question, contexts, answer):
    """
    RAG 시스템의 테스트 결과를 확인하기 위한 함수입니다.

    실제 RAG에서는
    contexts = Retriever가 검색한 문서
    answer = LLM이 생성한 답변
    """

    print("=" * 60)
    print("질문")
    print("=" * 60)
    print(question)

    print("\n" + "=" * 60)
    print("검색된 Context")
    print("=" * 60)

    for i, context in enumerate(contexts, 1):
        print(f"\n[Context {i}]")
        print(context)

    print("\n" + "=" * 60)
    print("RAG 답변")
    print("=" * 60)
    print(answer)

In [19]:
#10. 직접 RAG 테스트 실행
test_rag(
    question=questions[0],
    contexts=contexts[0],
    answer=answers[0]
)

질문
주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?

검색된 Context

[Context 1]
주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. 기업의 수익성 대비 주가 수준을 평가하는 지표입니다.

[Context 2]
EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.

[Context 3]
업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다.

RAG 답변
PER(Price Earnings Ratio)은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다. PER이 낮으면 일반적으로 이익에 비해 주가가 낮은 것으로 해석할 수 있지만, 업종 특성과 기업의 성장성 등을 함께 고려해야 합니다.


In [20]:
#11. 일부러 문제가 있는 RAG를 만들어보자

bad_questions = [
    "주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?"
]

bad_contexts = [[
    "삼성전자는 국내 대표적인 반도체 기업입니다.",
    "반도체 산업은 메모리와 시스템 반도체 등 다양한 분야로 구성됩니다."
]]

bad_answers = [
    "PER은 주가를 주당순이익으로 나눈 값입니다. "
    "일반적으로 PER이 낮으면 저평가, 높으면 고평가로 볼 수 있습니다."
]

bad_references = [
    "PER은 현재 주가를 주당순이익(EPS)으로 나눈 값입니다."
]

In [21]:
#12. 문제가 있는 RAG를 직접 테스트
test_rag(
    question=bad_questions[0],
    contexts=bad_contexts[0],
    answer=bad_answers[0]
)

질문
주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?

검색된 Context

[Context 1]
삼성전자는 국내 대표적인 반도체 기업입니다.

[Context 2]
반도체 산업은 메모리와 시스템 반도체 등 다양한 분야로 구성됩니다.

RAG 답변
PER은 주가를 주당순이익으로 나눈 값입니다. 일반적으로 PER이 낮으면 저평가, 높으면 고평가로 볼 수 있습니다.


In [22]:
#13. 문제가 있는 RAG를 RAGAS로 평가
bad_data = {
    "user_input": bad_questions,
    "response": bad_answers,
    "retrieved_contexts": bad_contexts,
    "reference": bad_references
}

bad_dataset = Dataset.from_dict(bad_data)

In [23]:
bad_result = evaluate(
    dataset=bad_dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False
)

bad_df = bad_result.to_pandas()

display(
    bad_df[
        [
            "faithfulness",
            "answer_relevancy",
            "context_precision",
            "context_recall"
        ]
    ]
)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

,faithfulness,answer_relevancy,context_precision,context_recall
0,0.0,0.490936,0.0,0.0


In [24]:
#14. RAG 개선
improved_contexts = [[
    "주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. "
    "기업의 수익성 대비 주가 수준을 평가하는 지표입니다.",

    "EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.",

    "업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 "
    "개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다."
]]

improved_answers = [
    "PER(Price Earnings Ratio)은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. "
    "EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다. "
    "따라서 PER은 기업의 이익에 비해 주가가 어느 정도 수준인지 확인하는 데 활용할 수 있습니다."
]

In [25]:
#15. 개선된 RAG를 직접 테스트
test_rag(
    question=questions[0],
    contexts=improved_contexts[0],
    answer=improved_answers[0]
)

질문
주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?

검색된 Context

[Context 1]
주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. 기업의 수익성 대비 주가 수준을 평가하는 지표입니다.

[Context 2]
EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.

[Context 3]
업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다.

RAG 답변
PER(Price Earnings Ratio)은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다. 따라서 PER은 기업의 이익에 비해 주가가 어느 정도 수준인지 확인하는 데 활용할 수 있습니다.


In [26]:
#16. 개선된 RAG를 RAGAS로 평가
improved_data = {
    "user_input": questions,
    "response": improved_answers,
    "retrieved_contexts": improved_contexts,
    "reference": ground_truths
}

improved_dataset = Dataset.from_dict(improved_data)

In [27]:
improved_result = evaluate(
    dataset=improved_dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False
)

improved_df = improved_result.to_pandas()

display(
    improved_df[
        [
            "faithfulness",
            "answer_relevancy",
            "context_precision",
            "context_recall"
        ]
    ]
)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

,faithfulness,answer_relevancy,context_precision,context_recall
0,1.0,0.652216,1.0,1.0


In [29]:
#17. 개선 전/후 비교
metric_names = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

comparison = pd.DataFrame({
    "문제 상황": bad_df[metric_names].mean(),
    "개선 후": improved_df[metric_names].mean()
})

comparison["변화"] = (
    comparison["개선 후"] - comparison["문제 상황"]
)

display(comparison)

,문제 상황,개선 후,변화
faithfulness,0.000000,1.000000,1.000000
answer_relevancy,0.490936,0.652216,0.161281
context_precision,0.000000,1.000000,1.000000
context_recall,0.000000,1.000000,1.000000


In [30]:
#18. 여러 질문으로 확장
test_questions = [
    "주가수익비율(PER)이란 무엇인가요?",
    "EPS란 무엇인가요?",
    "PER은 어떻게 계산하나요?",
    "업종 평균 PER은 왜 비교하나요?",
]

test_references = [
    "PER은 현재 주가를 주당순이익(EPS)으로 나눈 값입니다.",
    "EPS는 기업의 순이익을 발행 주식 수로 나눈 주당순이익입니다.",
    "PER은 현재 주가를 주당순이익(EPS)으로 나누어 계산합니다.",
    "업종 평균 PER은 동일 업종 기업과 비교하여 상대적인 주가 수준을 판단하는 데 활용합니다."
]

In [31]:
print(type(contexts))
print(len(contexts))

print("\ncontexts[0]의 타입:")
print(type(contexts[0]))

print("\ncontexts[0]의 길이:")
print(len(contexts[0]))

print("\n각 Context 확인:")
for i, context in enumerate(contexts[0], 1):
    print(f"\nContext {i}:")
    print(context)

<class 'list'>
1

contexts[0]의 타입:
<class 'list'>

contexts[0]의 길이:
3

각 Context 확인:

Context 1:
주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. 기업의 수익성 대비 주가 수준을 평가하는 지표입니다.

Context 2:
EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.

Context 3:
업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다.


In [32]:
test_rag(
    question=questions[0],
    contexts=contexts[0],
    answer=answers[0]
)

질문
주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?

검색된 Context

[Context 1]
주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. 기업의 수익성 대비 주가 수준을 평가하는 지표입니다.

[Context 2]
EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.

[Context 3]
업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다.

RAG 답변
PER(Price Earnings Ratio)은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다. PER이 낮으면 일반적으로 이익에 비해 주가가 낮은 것으로 해석할 수 있지만, 업종 특성과 기업의 성장성 등을 함께 고려해야 합니다.


In [41]:
#1. 평가 데이터의 기본 구조 만들기
#테스트 질문과 기준 답변
test_questions = [
    "주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?",
    "EPS란 무엇인가요?",
    "업종 평균 PER은 왜 비교하나요?"
]

test_references = [
    "PER은 현재 주가를 주당순이익(EPS)으로 나눈 값으로, "
    "기업의 주가가 이익에 비해 어느 정도 수준인지 나타내는 지표입니다.",

    "EPS는 기업의 순이익을 발행 주식 수로 나눈 주당순이익입니다.",

    "업종 평균 PER은 동일 업종의 기업들과 비교하여 "
    "개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다."
]

In [43]:
#2. RAG 결과를 저장하는 함수 만들기
def make_evaluation_record(
    question,
    contexts,
    answer,
    reference
):
    """
    RAG 실행 결과를 RAGAS 평가용 데이터 형식으로 저장합니다.
    """

    return {
        "user_input": question,
        "retrieved_contexts": contexts,
        "response": answer,
        "reference": reference
    }

In [44]:
#3. 여러 질문의 결과를 한 번에 저장하는 함수
def make_evaluation_dataset(
    questions,
    contexts_list,
    answers,
    references
):
    """
    여러 질문에 대한 RAG 결과를
    RAGAS 평가용 데이터셋 형태로 만듭니다.
    """

    records = []

    for question, contexts, answer, reference in zip(
        questions,
        contexts_list,
        answers,
        references
    ):
        record = make_evaluation_record(
            question=question,
            contexts=contexts,
            answer=answer,
            reference=reference
        )

        records.append(record)

    return records

In [45]:
#4. 현재 RAG 결과를 저장해보기
current_contexts = [
    [
        "주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. "
        "기업의 수익성 대비 주가 수준을 평가하는 지표입니다.",

        "EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.",

        "업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 "
        "개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다."
    ],

    [
        "EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다."
    ],

    [
        "업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 평균낸 값으로, "
        "개별 기업의 상대적인 가치를 판단하는 기준이 됩니다."
    ]
]

In [46]:
current_answers = [
    "PER은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. "
    "EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다.",

    "EPS는 기업의 당기순이익을 발행주식 총수로 나눈 주당순이익입니다.",

    "업종 평균 PER은 같은 산업에 속한 기업들의 PER과 비교하여 "
    "개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다."
]

In [47]:
#5. RAGAS 평가 데이터 생성

current_records = make_evaluation_dataset(
    questions=test_questions,
    contexts_list=current_contexts,
    answers=current_answers,
    references=test_references
)

current_records

[{'user_input': '주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?',
  'retrieved_contexts': ['주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. 기업의 수익성 대비 주가 수준을 평가하는 지표입니다.',
   'EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.',
   '업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다.'],
  'response': 'PER은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다.',
  'reference': 'PER은 현재 주가를 주당순이익(EPS)으로 나눈 값으로, 기업의 주가가 이익에 비해 어느 정도 수준인지 나타내는 지표입니다.'},
 {'user_input': 'EPS란 무엇인가요?',
  'retrieved_contexts': ['EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.'],
  'response': 'EPS는 기업의 당기순이익을 발행주식 총수로 나눈 주당순이익입니다.',
  'reference': 'EPS는 기업의 순이익을 발행 주식 수로 나눈 주당순이익입니다.'},
 {'user_input': '업종 평균 PER은 왜 비교하나요?',
  'retrieved_contexts': ['업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 평균낸 값으로, 개별 기업의 상대적인 가치를 판단하는 기준이 됩니다.'],
  'response': '업종 평균 PER은 같은 산업에 속한 기업들의 PER과 비교하여 개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다.',
  'reference': '업종 평균 PER은 동일 업종의 기업들과 비교하여 개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다.'}]

In [49]:
[
    {
        "user_input": "주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?",
        "retrieved_contexts": [
            "PER 설명...",
            "EPS 설명...",
            "업종 평균 PER 설명..."
        ],
        "response": "PER은 현재 주가를...",
        "reference": "PER은 현재 주가를..."
    },

    {
        "user_input": "EPS란 무엇인가요?",
        "retrieved_contexts": [
            "EPS 설명..."
        ],
        "response": "EPS는...",
        "reference": "EPS는..."
    }
]

[{'user_input': '주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?',
  'retrieved_contexts': ['PER 설명...', 'EPS 설명...', '업종 평균 PER 설명...'],
  'response': 'PER은 현재 주가를...',
  'reference': 'PER은 현재 주가를...'},
 {'user_input': 'EPS란 무엇인가요?',
  'retrieved_contexts': ['EPS 설명...'],
  'response': 'EPS는...',
  'reference': 'EPS는...'}]

In [50]:
#6. Dataset으로 변환
from datasets import Dataset

current_dataset = Dataset.from_list(current_records)

display(current_dataset.to_pandas())

,user_input,retrieved_contexts,response,reference
0,주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?,"[주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. 기업의 수익성 대비 주가 수준을 평가하는 지표입니다., EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다., 업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다.]",PER은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다.,"PER은 현재 주가를 주당순이익(EPS)으로 나눈 값으로, 기업의 주가가 이익에 비해 어느 정도 수준인지 나타내는 지표입니다."
1,EPS란 무엇인가요?,[EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.],EPS는 기업의 당기순이익을 발행주식 총수로 나눈 주당순이익입니다.,EPS는 기업의 순이익을 발행 주식 수로 나눈 주당순이익입니다.
2,업종 평균 PER은 왜 비교하나요?,"[업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 평균낸 값으로, 개별 기업의 상대적인 가치를 판단하는 기준이 됩니다.]",업종 평균 PER은 같은 산업에 속한 기업들의 PER과 비교하여 개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다.,업종 평균 PER은 동일 업종의 기업들과 비교하여 개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다.


In [51]:
#7. RAGAS 평가 함수도 만들어두자
def evaluate_rag(records):
    """
    RAGAS 평가용 records를 받아
    RAGAS 평가 결과를 DataFrame으로 반환합니다.
    """

    dataset = Dataset.from_list(records)

    result = evaluate(
        dataset=dataset,
        metrics=[
            context_precision,
            context_recall,
            faithfulness,
            answer_relevancy,
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        raise_exceptions=False
    )

    df = result.to_pandas()

    return df

In [52]:
#8. 현재 버전 평가
df_v1 = evaluate_rag(current_records)

display(
    df_v1[
        [
            "user_input",
            "faithfulness",
            "answer_relevancy",
            "context_precision",
            "context_recall"
        ]
    ]
)

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,faithfulness,answer_relevancy,context_precision,context_recall
0,주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?,1.0,0.505239,1.0,1.0
1,EPS란 무엇인가요?,1.0,0.848641,1.0,1.0
2,업종 평균 PER은 왜 비교하나요?,1.0,0.853326,1.0,1.0


In [53]:
#9. V1 결과를 저장해두기
v1_records = current_records

v1_dataset = Dataset.from_list(v1_records)

display(v1_dataset.to_pandas())

,user_input,retrieved_contexts,response,reference
0,주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?,"[주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다. 기업의 수익성 대비 주가 수준을 평가하는 지표입니다., EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다., 업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 비교하여 개별 기업의 상대적인 주가 수준을 판단하는 데 활용할 수 있습니다.]",PER은 현재 주가를 주당순이익(EPS)으로 나눈 비율입니다. EPS는 기업의 순이익을 발행 주식 수로 나눈 값입니다.,"PER은 현재 주가를 주당순이익(EPS)으로 나눈 값으로, 기업의 주가가 이익에 비해 어느 정도 수준인지 나타내는 지표입니다."
1,EPS란 무엇인가요?,[EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.],EPS는 기업의 당기순이익을 발행주식 총수로 나눈 주당순이익입니다.,EPS는 기업의 순이익을 발행 주식 수로 나눈 주당순이익입니다.
2,업종 평균 PER은 왜 비교하나요?,"[업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 평균낸 값으로, 개별 기업의 상대적인 가치를 판단하는 기준이 됩니다.]",업종 평균 PER은 같은 산업에 속한 기업들의 PER과 비교하여 개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다.,업종 평균 PER은 동일 업종의 기업들과 비교하여 개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다.


In [54]:
v1_dataset.to_json(
    "/content/rag_evaluation_v1.json",
    orient="records"
)

print("V1 평가 데이터 저장 완료")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

V1 평가 데이터 저장 완료


In [55]:
v2_contexts = [
    [
        "주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다.",
        "EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다."
    ],

    [
        "EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다."
    ],

    [
        "업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 평균낸 값입니다.",
        "동일 업종의 기업들과 비교하여 개별 기업의 상대적인 주가 수준을 판단할 수 있습니다."
    ]
]

v2_answers = [
    "PER은 현재 주가를 주당순이익(EPS)으로 나눈 값입니다. "
    "기업의 주가가 이익에 비해 어느 정도 수준인지 판단하는 데 활용할 수 있습니다.",

    "EPS는 기업의 당기순이익을 발행주식 총수로 나눈 주당순이익입니다.",

    "업종 평균 PER은 동일 업종의 기업들과 비교하여 "
    "개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다."
]

In [56]:
#11. V2 평가 데이터 만들기
v2_records = make_evaluation_dataset(
    questions=test_questions,
    contexts_list=v2_contexts,
    answers=v2_answers,
    references=test_references
)

display(pd.DataFrame(v2_records))

,user_input,retrieved_contexts,response,reference
0,주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?,"[주가수익비율(PER)은 현재 주가를 1주당 순이익(EPS)으로 나눈 수치입니다., EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.]",PER은 현재 주가를 주당순이익(EPS)으로 나눈 값입니다. 기업의 주가가 이익에 비해 어느 정도 수준인지 판단하는 데 활용할 수 있습니다.,"PER은 현재 주가를 주당순이익(EPS)으로 나눈 값으로, 기업의 주가가 이익에 비해 어느 정도 수준인지 나타내는 지표입니다."
1,EPS란 무엇인가요?,[EPS(Earnings Per Share)는 기업의 당기순이익을 발행주식 총수로 나눈 값입니다.],EPS는 기업의 당기순이익을 발행주식 총수로 나눈 주당순이익입니다.,EPS는 기업의 순이익을 발행 주식 수로 나눈 주당순이익입니다.
2,업종 평균 PER은 왜 비교하나요?,"[업종 평균 PER은 동일 산업군에 속한 기업들의 PER을 평균낸 값입니다., 동일 업종의 기업들과 비교하여 개별 기업의 상대적인 주가 수준을 판단할 수 있습니다.]",업종 평균 PER은 동일 업종의 기업들과 비교하여 개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다.,업종 평균 PER은 동일 업종의 기업들과 비교하여 개별 기업의 상대적인 주가 수준을 판단하기 위해 사용합니다.


In [57]:
#12. V2 RAGAS 재평가
df_v2 = evaluate_rag(v2_records)

display(
    df_v2[
        [
            "user_input",
            "faithfulness",
            "answer_relevancy",
            "context_precision",
            "context_recall"
        ]
    ]
)

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,faithfulness,answer_relevancy,context_precision,context_recall
0,주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?,1.0,0.510183,1.0,1.0
1,EPS란 무엇인가요?,1.0,0.697309,1.0,1.0
2,업종 평균 PER은 왜 비교하나요?,1.0,0.853326,1.0,1.0


In [58]:
#13. V1과 V2 평균 점수 비교
metric_names = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

comparison = pd.DataFrame({
    "V1": df_v1[metric_names].mean(),
    "V2": df_v2[metric_names].mean()
})

comparison["변화"] = comparison["V2"] - comparison["V1"]

display(comparison)

,V1,V2,변화
faithfulness,1.000000,1.00000,0.000000e+00
answer_relevancy,0.735735,0.68694,-4.879564e-02
context_precision,1.000000,1.00000,1.666667e-11
context_recall,1.000000,1.00000,0.000000e+00


In [ ]:
#14. V2 이후에도 계속 반복할 수 있다
v3_records = make_evaluation_dataset(
    questions=test_questions,
    contexts_list=v3_contexts,
    answers=v3_answers,
    references=test_references
)

df_v3 = evaluate_rag(v3_records)

In [61]:
comparison = pd.DataFrame({
    "V1": df_v1[metric_names].mean(),
    "V2": df_v2[metric_names].mean(),
#    "V3": df_v3[metric_names].mean()
})

display(comparison)

,V1,V2
faithfulness,1.000000,1.00000
answer_relevancy,0.735735,0.68694
context_precision,1.000000,1.00000
context_recall,1.000000,1.00000


In [62]:
#15. 더 좋은 방법: 버전까지 저장하기
all_records = []

In [63]:
#V1
for record in v1_records:
    record["version"] = "V1"

all_records.extend(v1_records)

In [64]:
#V2
for record in v2_records:
    record["version"] = "V2"

all_records.extend(v2_records)

In [ ]:
#V3
for record in v3_records:
    record["version"] = "V3"

all_records.extend(v3_records)

In [ ]:
#실제 RAG에서는 이 부분만 Retriever와 LLM으로 대체
records = []

for question, reference in zip(
    test_questions,
    test_references
):

    # ① Retriever가 실제로 검색
    retrieved_docs = retriever.invoke(question)

    contexts = [
        doc.page_content
        for doc in retrieved_docs
    ]

    # ② LLM이 실제로 답변 생성
    answer = rag_chain.invoke(question)

    # ③ RAGAS 평가용 데이터 저장
    records.append({
        "version": "V1",
        "user_input": question,
        "retrieved_contexts": contexts,
        "response": answer,
        "reference": reference
    })

In [33]:
questions = ["주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?"]
ground_truths = ["주가수익비율(PER)은 기업의 주가를 주당순이익(EPS)으로 나눈 값으로, 해당 기업의 주가가 실제 수익에 비해 얼마나 높게 형성되어 있는지를 나타내는 지표입니다. 이는 투자자들이 기업의 이익 1원을 얻기 위해 지불하는 비용을 의미하며, 보통 업종 평균 PER과 비교하여 저평가 혹은 고평가 여부를 판단합니다."]
answers = ["PER(Price Earnings Ratio)은 현재 주가를 주당순이익(EPS)으로 나눈 비율을 의미합니다. 여기서 주가는 시장에서 거래되는 가격을 말하며, 주당순이익(EPS)은 기업이 벌어들인 순이익을 발행 주식 수로 나눈 값입니다. PER이 낮을수록 해당 기업이 벌어들이는 이익에 비해 주가가 저렴하다는 '저평가' 신호로 해석될 수 있으며, 반대로 높으면 '고평가' 혹은 미래 성장성이 높다고 판단합니다."]
contexts = [[
    "A: 주가수익비율(PER)은 현재 주가를 1주당 예상 순이익(EPS)으로 나눈 수치로, 기업의 수익성 대비 주가 수준을 평가하는 핵심 지표입니다. "
    "* EPS(Earnings Per Share): 기업이 벌어들인 당기순이익을 발행주식 총수로 나눈 값. "
    "** 업종 평균 PER: 동일 산업군 내 기업들의 PER을 평균낸 값으로, 개별 기업의 상대적 가치를 판단하는 기준이 됨. "
    "\n\n"
    "| 종목명 | 현재가 | EPS | PER | 업종평균 PER | 투자판단 |\n"
    "| :--- | :--- | :--- | :--- | :--- | :--- |\n"
    "| 삼성전자 | 75,000 | 5,000 | 15.0x | 18.2x | 저평가 |\n"
    "| SK하이닉스 | 180,000 | 9,000 | 20.0x | 18.2x | 고평가(성장세) |\n"
    "| 현대차 | 250,000 | 35,000 | 7.1x | 10.5x | 저평가 |\n"
    "| 카카오 | 50,000 | 500 | 100.0x | 45.0x | 고평가 |"
]]

In [34]:
from datasets import Dataset
import pandas as pd

# To dict
data = {
  "user_input": questions,
  "response": answers,
  "retrieved_contexts": contexts,
  "reference": ground_truths
}

# Convert dict to dataset
dataset = Dataset.from_dict(data)

In [35]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

/tmp/ipykernel_645/2261505146.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_645/2261505146.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_645/2261505146.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_645/2261505146.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.

In [36]:
# 평가자(Judge) 역할을 할 LLM 과 임베딩을 정합니다
# 지정하지 않으면 RAGAS 가 고른 기본 임베딩이 위 지표들과 맞지 않아 answer_relevancy 가 계산되지 않습니다
evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
evaluator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [37]:
result = evaluate(
    dataset=dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

In [38]:
# score 출력
print(result)
# DataFrame 생성
df = result.to_pandas()
df

{'context_precision': 1.0000, 'context_recall': 1.0000, 'faithfulness': 0.8000, 'answer_relevancy': 0.5984}


,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,주가수익비율(PER)이란 무엇이며 어떻게 계산하나요?,"[A: 주가수익비율(PER)은 현재 주가를 1주당 예상 순이익(EPS)으로 나눈 수치로, 기업의 수익성 대비 주가 수준을 평가하는 핵심 지표입니다. * EPS(Earnings Per Share): 기업이 벌어들인 당기순이익을 발행주식 총수로 나눈 값. ** 업종 평균 PER: 동일 산업군 내 기업들의 PER을 평균낸 값으로, 개별 기업의 상대적 가치를 판단하는 기준이 됨. \n\n| 종목명 | 현재가 | EPS | PER | 업종평균 PER | 투자판단 |\n| :--- | :--- | :--- | :--- | :--- | :--- |\n| 삼성전자 | 75,000 | 5,000 | 15.0x | 18.2x | 저평가 |\n| SK하이닉스 | 180,000 | 9,000 | 20.0x | 18.2x | 고평가(성장세) |\n| 현대차 | 250,000 | 35,000 | 7.1x | 10.5x | 저평가 |\n| 카카오 | 50,000 | 500 | 100.0x | 45.0x | 고평가 |]","PER(Price Earnings Ratio)은 현재 주가를 주당순이익(EPS)으로 나눈 비율을 의미합니다. 여기서 주가는 시장에서 거래되는 가격을 말하며, 주당순이익(EPS)은 기업이 벌어들인 순이익을 발행 주식 수로 나눈 값입니다. PER이 낮을수록 해당 기업이 벌어들이는 이익에 비해 주가가 저렴하다는 '저평가' 신호로 해석될 수 있으며, 반대로 높으면 '고평가' 혹은 미래 성장성이 높다고 판단합니다.","주가수익비율(PER)은 기업의 주가를 주당순이익(EPS)으로 나눈 값으로, 해당 기업의 주가가 실제 수익에 비해 얼마나 높게 형성되어 있는지를 나타내는 지표입니다. 이는 투자자들이 기업의 이익 1원을 얻기 위해 지불하는 비용을 의미하며, 보통 업종 평균 PER과 비교하여 저평가 혹은 고평가 여부를 판단합니다.",1.0,1.0,0.8,0.59838


In [39]:
# 평가용 가상 데이터셋
# 열 이름은 RAGAS 표준 이름(user_input / retrieved_contexts / response / reference)을 씁니다. 앞의 PER 데이터셋과 같습니다
data_samples = {
    "user_input": [
        "프랑스의 수도는 어디이며, 가장 유명한 미술관은 무엇인가요?",
        "블랙홀의 사건의 지평선이란 무엇인가요?"
    ],
    "retrieved_contexts": [
        [
            "파리는 프랑스의 수도이자 세계적인 문화 중심지입니다.",
            "루브르 박물관은 파리에 위치해 있으며, 모나리자와 같은 명작을 소장하고 있습니다."
        ],
        [
            "블랙홀은 중력이 너무 강해 빛조차 빠져나갈 수 없는 천체입니다.",
            "우주 공간에는 수많은 블랙홀이 존재한다고 추정됩니다." # 의도적 누락: 사건의 지평선 설명 없음
        ]
    ],
    "response": [
        "프랑스의 수도는 파리이며, 가장 유명한 미술관은 루브르 박물관입니다.",
        "블랙홀의 사건의 지평선은 중력이 너무 강해 빛조차 빠져나갈 수 없는 경계면을 의미합니다. 일반 상대성 이론에 의해 예측되었습니다." # RAG 관점의 환각 발생
    ],
    "reference": [
        "프랑스의 수도는 파리이고, 대표적인 미술관은 루브르 박물관입니다.",
        "사건의 지평선(Event Horizon)은 블랙홀의 탈출 속도가 빛의 속도를 넘어서는 공간의 경계입니다."
    ]
}

evaluation_dataset = Dataset.from_dict(data_samples)

In [40]:
# 결과 평가
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False # API 에러 발생 시 중단하지 않음
)

pd.set_option('display.max_colwidth', None)
df_result = result.to_pandas()
display(df_result[['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']])

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

,faithfulness,answer_relevancy,context_precision,context_recall
0,0.500000,0.54465,0.5,1.0
1,0.666667,1.00000,1.0,0.0
